In [ ]:
#import open cv

import open_

gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

# Extra research

https://docs.opencv.org/4.x/d7/d4d/tutorial_py_thresholding.html?

Thresholding in simple terms



In [ ]:
import cv2
import numpy as np
import tensorflow as tf
from collections import Counter

# ----------------------------
# SETTINGS
# ----------------------------
MODEL_PATH = "rice_classifier.keras"   # your saved model
IMAGE_PATH = "handful.jpg"             # image to test
IMG_SIZE = 224                         # model input size
MIN_AREA = 100                         # ignore tiny noise
CLASS_NAMES = ["basmati", "jasmine", "brown", "sushi"]  # change to your classes

# ----------------------------
# LOAD MODEL
# ----------------------------
model = tf.keras.models.load_model(MODEL_PATH)

# ----------------------------
# LOAD IMAGE
# ----------------------------
image = cv2.imread(IMAGE_PATH)
if image is None:
    raise FileNotFoundError(f"Could not read image: {IMAGE_PATH}")

original = image.copy()

# ----------------------------
# PREPROCESS FOR SEGMENTATION
# ----------------------------
gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
blur = cv2.GaussianBlur(gray, (5, 5), 0)

# Otsu threshold
_, mask = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

# If foreground/background are flipped, invert
white_ratio = np.mean(mask == 255)
if white_ratio > 0.7:
    mask = cv2.bitwise_not(mask)

# Clean small noise
kernel = np.ones((3, 3), np.uint8)
mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=1)
mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=1)

# ----------------------------
# FIND CONTOURS
# ----------------------------
contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

results = []
counts = Counter()

# ----------------------------
# PROCESS EACH CONTOUR
# ----------------------------
for i, contour in enumerate(contours):
    area = cv2.contourArea(contour)
    if area < MIN_AREA:
        continue

    x, y, w, h = cv2.boundingRect(contour)

    # Crop image and corresponding mask
    crop = original[y:y+h, x:x+w]
    crop_mask = mask[y:y+h, x:x+w]

    # Apply mask so background becomes black
    masked_crop = cv2.bitwise_and(crop, crop, mask=crop_mask)

    # Convert to grayscale because your training images were grayscale
    grain_gray = cv2.cvtColor(masked_crop, cv2.COLOR_BGR2GRAY)

    # Resize to match model input
    grain_resized = cv2.resize(grain_gray, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)

    # Normalize
    grain_resized = grain_resized.astype(np.float32) / 255.0

    # Add batch and channel dimensions: (1, H, W, 1)
    x_input = np.expand_dims(grain_resized, axis=(0, -1))

    # Predict
    probs = model.predict(x_input, verbose=0)[0]
    pred_idx = int(np.argmax(probs))
    pred_label = CLASS_NAMES[pred_idx]
    confidence = float(probs[pred_idx])

    counts[pred_label] += 1
    results.append({
        "box": (x, y, w, h),
        "label": pred_label,
        "confidence": confidence
    })

    # Draw result on image
    cv2.rectangle(original, (x, y), (x+w, y+h), (0, 255, 0), 2)
    cv2.putText(
        original,
        f"{pred_label} {confidence:.2f}",
        (x, max(20, y - 10)),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.5,
        (0, 255, 0),
        1,
        cv2.LINE_AA
    )

# ----------------------------
# OUTPUT
# ----------------------------
print("Detections:")
for r in results:
    print(r)

print("\nCounts:")
print(dict(counts))

cv2.imwrite("mask.png", mask)
cv2.imwrite("annotated.png", original)

print("\nSaved:")
print("- mask.png")
print("- annotated.png")